# SIFT Landmark Recognition & Self-Correcting Rotation Pipeline

Notebook này được thiết kế để chạy quy trình huấn luyện và nhận diện địa danh sử dụng SIFT, Bag of Visual Words (BoVW), Spatial Pyramid Matching (SPM) và SVM trên **Google Colab**.

## Thành phần hệ thống:
1. **Trích xuất đặc trưng**: SIFT trích xuất các điểm đặc trưng.
2. **Tạo từ điển thị giác**: Gom cụm SIFT descriptors bằng K-Means.
3. **Tạo Histogram & SPM**: Phân vùng không gian 1x1 và 2x2 để giữ thông tin cấu trúc.
4. **Phân loại**: Train SVM Classifier.
5. **Tự xoay ảnh (Self-Correcting)**: SIFT matching đối chiếu chéo ảnh mẫu, tính góc lệch bằng Orientation Histogram Voting + RANSAC, xoay ngược lại để khôi phục hướng thẳng đứng trước khi nhận diện.

## Bước 1: Tải dự án & Kích hoạt Git LFS (Large File Storage)
Tải dự án từ GitHub và kích hoạt Git LFS để tải về toàn bộ các file ảnh thực tế (thay vì các file con trỏ văn bản nhỏ).

In [ ]:
# 1. Tải dự án từ GitHub
!git clone https://github.com/hnihTyoB/KhaiPhaDuLieuDPT.git

# 2. Di chuyển vào thư mục dự án
%cd KhaiPhaDuLieuDPT

# 3. Khởi tạo và tải về toàn bộ ảnh thực tế từ Git LFS
!git lfs install
!git lfs pull

## Bước 2: Cài đặt thư viện
Cài đặt toàn bộ các thư viện cần thiết trực tiếp từ tệp cấu hình `requirements.txt` có sẵn của dự án.

In [ ]:
# Cài đặt các thư viện từ file requirements.txt
!pip install -r requirements.txt

## Bước 3: Kết nối Google Drive & Sao chép dữ liệu (Dataset)
Thực hiện kết nối trực tiếp với Google Drive cá nhân của bạn để sao chép file `dataset.zip` từ mục `MyDrive/` vào môi trường chạy và tự động giải nén.

**⚠️ Hướng dẫn lưu trữ quan trọng trên Google Drive:**
1. Tải bộ dữ liệu `dataset.zip` theo đường dẫn liên kết chia sẻ trong tài liệu dự án về máy tính của bạn.
2. Mở **Google Drive** cá nhân của bạn lên.
3. Tải file `dataset.zip` lên và đặt trực tiếp ở **thư mục gốc** của Drive (mục **Drive của tôi** / **My Drive**, không được đặt bên trong bất kỳ thư mục con nào khác).
4. Điều này giúp câu lệnh sao chép `!cp /content/drive/MyDrive/dataset.zip .` ở ô code bên dưới tìm thấy file và sao chép chính xác.

In [ ]:
import os
import zipfile

zip_path = 'dataset.zip'

# 1. Kết nối Google Drive và sao chép file dataset.zip nếu chưa có sẵn dữ liệu
if not os.path.exists('dataset') and not os.path.exists(zip_path):
    print("Đang tiến hành kết nối Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    print("Đang sao chép file dataset.zip từ Drive...")
    !cp /content/drive/MyDrive/dataset.zip .

# 2. Tiến hành giải nén file dataset.zip
if os.path.exists(zip_path) and not os.path.exists('dataset'):
    print("✓ Tìm thấy file dataset.zip. Đang tiến hành giải nén...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Giải nén hoàn tất.")
elif os.path.exists('dataset') and len(os.listdir('dataset')) > 0:
    print("✓ Thư mục 'dataset' đã có sẵn dữ liệu. Bỏ qua sao chép và giải nén.")

# 3. Hiển thị kiểm tra các thư mục con trong dataset/
if os.path.exists('dataset') and len(os.listdir('dataset')) > 0:
    classes = [d for d in os.listdir('dataset') if os.path.isdir(os.path.join('dataset', d))]
    print(f"--> Đã sẵn sàng. Phát hiện {len(classes)} lớp địa danh: {classes}")
else:
    print("--> Cảnh báo: Chưa tìm thấy thư mục 'dataset' hợp lệ. Hãy kiểm tra lại file zip của bạn trên Google Drive.")

## Bước 4: Huấn luyện mô hình (Training Model)
Gọi hàm `train_pipeline` từ `landmark_sift_bovw.py` để trích xuất SIFT -> build K-Means từ vựng thị giác -> tạo histogram đặc trưng -> train SVM.

In [ ]:
from landmark_sift_bovw import train_pipeline

# Chạy tiến trình huấn luyện mô hình
success = train_pipeline()
if success:
    print("Mô hình đã được huấn luyện và lưu thành công trong thư mục 'models/'")
else:
    print("Huấn luyện thất bại. Vui lòng kiểm tra lại cấu trúc thư mục dataset.")

## Bước 5: Nhập ảnh kiểm thử thủ công và Dự đoán (Prediction)
Hãy nhập thủ công đường dẫn của ảnh bạn muốn nhận diện (ví dụ một ảnh bất kỳ trong thư mục `test_images/` như `test_images/image1.png`).

In [ ]:
import os
from landmark_sift_bovw import predict_image

# Nhập đường dẫn ảnh thủ công bằng tay
image_path = input("Nhập đường dẫn ảnh kiểm thử (ví dụ: test_images/image1.png): ").strip().strip('"').strip("'")

if os.path.exists(image_path):
    print(f"Đang chạy dự đoán cho ảnh: {image_path}")
    predict_image(image_path)
else:
    print(f"Lỗi: Không tìm thấy ảnh tại đường dẫn: {image_path}")

## Bước 6: Chạy quy trình Tự chỉnh góc xoay & Nhận dạng (Self-Correcting Pipeline)
Quy trình tự động chỉnh xoay ảnh bằng SIFT trước khi nhận dạng. Hệ thống sẽ cho phép bạn giả lập xoay nghiêng ảnh đi một góc nhất định để xem khả năng tự khôi phục hướng thẳng đứng và nhận diện chính xác của hệ thống.

In [ ]:
from sift_landmark_pipeline import full_pipeline

if 'image_path' in locals() and os.path.exists(image_path):
    # Cho phép bạn giả lập xoay ảnh kiểm thử đi một góc nhất định
    sim = input("Giả lập xoay? Nhập góc lệch (hoặc nhấn Enter để bỏ qua): ").strip()
    simulate_angle = None
    if sim:
        try:
            simulate_angle = float(sim)
        except ValueError:
            print("[WARN] Góc không hợp lệ, bỏ qua giả lập xoay.")
            
    print(f"\nChạy Pipeline với ảnh: {image_path}")
    full_pipeline(image_path, simulate_angle)
else:
    print("Lỗi: Hãy chạy ô nhập ảnh ở Bước 5 trước hoặc đảm bảo đường dẫn ảnh hợp lệ.")

## Bước 7: Trực quan hóa kết quả so sánh
Hiển thị biểu đồ và ảnh so sánh trực tiếp kết quả trước và sau khi được hệ thống tự xoay chỉnh góc.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from IPython.display import Image, display

# 1. Hiển thị ảnh so sánh quy trình
comparison_path = 'results_pipeline/pipeline_comparison.png'
if os.path.exists(comparison_path):
    print("\n--- ẢNH SO SÁNH KẾT QUẢ PIPELINE (ẢNH MẪU vs TRƯỚC CHỈNH vs SAU CHỈNH) ---")
    display(Image(filename=comparison_path, width=900))
else:
    print("Không tìm thấy ảnh pipeline_comparison.png")

# 2. Hiển thị ảnh khớp đặc trưng SIFT (SIFT Matches)
matches_path = 'results_pipeline/pipeline_matches.png'
if os.path.exists(matches_path):
    print("\n--- TRỰC QUAN HÓA CÁC ĐIỂM KHỚP SIFT (SIFT MATCHES) ---")
    img_matches = cv2.imread(matches_path)
    img_matches_rgb = cv2.cvtColor(img_matches, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 6))
    plt.imshow(img_matches_rgb)
    plt.axis('off')
    plt.show()
else:
    print("Không tìm thấy ảnh pipeline_matches.png")